# Reconstruct a saved native cluster

This self-contained example first runs the small synthetic demo with cluster
saving enabled, then reconstructs an accepted cluster. It no longer needs an
unprovided JSON cluster or a hard-coded time-delay filter rate.


In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

# Launch from this source checkout (or its examples directory).
candidates = [Path.cwd(), Path.cwd() / "pycwb", *Path.cwd().parents]
repo = next((p for p in candidates if (p / "examples/demo/user_parameters.yaml").is_file()), None)
if repo is None:
    raise RuntimeError("Open this notebook from the matching PycWB source checkout.")
sys.path.insert(0, str(repo / "examples"))
from notebook_support import prepare_injection_config
from pycwb.modules.logger import logger_init
logger_init()
%matplotlib inline


In [ ]:
from pycwb.workflow.run import search
from pycwb.modules.catalog import Catalog

work, config_path, config = prepare_injection_config("cluster-reconstruction", updates={"save_cluster": True})
run = work / "search"
search(str(config_path), working_dir=str(run))
cat = Catalog.open(str(run / "catalog/catalog.parquet"))
events = cat.triggers().to_pandas()
event = events.nlargest(1, "rho").iloc[0]
print(event[["id", "rho", "net_cc"]])


In [ ]:
# Use the run's serialized cluster and its matching saved configuration.
import gzip
import orjson
from dacite import from_dict, Config as DaciteConfig
from pycwb.types.network_cluster import Cluster
from pycwb.modules.reconstruction import get_network_MRA_wave

cluster_files = sorted((run / "trigger").rglob("cluster*.json*"))
print("Saved clusters:", cluster_files)
if not cluster_files:
    raise RuntimeError("No cluster was saved; inspect search completion and save_cluster.")
cluster_file = next(path for path in cluster_files if str(event["id"]).rsplit("_", 1)[-1] in str(path))
opener = gzip.open if cluster_file.suffix == ".gz" else open
with opener(cluster_file, "rb") as handle:
    cluster = from_dict(Cluster, orjson.loads(handle.read()),
                            config=DaciteConfig(type_hooks={np.ndarray: np.asarray}))
job = next(job for job in cat.jobs if job["index"] == event["job_id"])
waves = get_network_MRA_wave(config, cluster, config.rateANA, config.nIFO,
                             config.TDRate, "signal", 0, True, whiten=True,
                             start_time=job["analyze_start"] - job["seg_edge"])
fig, ax = plt.subplots()
for ifo, wave in zip(config.ifo, waves):
    ax.plot(wave.sample_times - config.injection["parameters"]["gps_time"], wave.data, label=ifo)
ax.set(xlabel="Time from injection [s]", ylabel="Reconstructed whitened strain")
ax.legend()


For ordinary inspection use the already saved `output/wave.h5` products, as shown in the event-inspection tutorial. Each dataset carries its own epoch and sample rate.
